In [ ]:
# -*- coding: utf-8 -*-
"""
Berramdane Model V9.4 
Author : Al Moalim Berramdane
"""

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Checkbox, IntSlider
from scipy.signal import find_peaks
import warnings
warnings.filterwarnings('ignore')

h, m, L_total = 6.626e-34, 9.109e-31, 2.2

# ========== إضافة حساب الزوايا (كما طلب ديبسيك) ==========
def compute_angles(v_mean, a_width, d_slit, L):
    lam = h / (m * v_mean)
    theta_i_rad = lam / d_slit
    theta_i_deg = theta_i_rad * 180 / np.pi
    theta_d_rad = lam / a_width
    theta_d_deg = theta_d_rad * 180 / np.pi
    return (theta_i_rad, theta_i_deg, theta_d_rad, theta_d_deg)

def de_broglie_wavelength(v): return h / (m * v)

def double_slit_intensity_single_velocity(x, v_par, L, a_width, d_slit):
    lam = de_broglie_wavelength(v_par)
    beta = (np.pi * d_slit * x) / (lam * L)
    alpha = (np.pi * a_width * x) / (lam * L)
    return np.cos(beta)**2 * np.sinc(alpha / np.pi)**2

def particle_like_pattern(x, v_par, L, a_width, d_slit):
    lam = de_broglie_wavelength(v_par)
    sigma = a_width * L / lam
    return 0.5 * (np.exp(-(x + d_slit/2)**2 / (2 * sigma**2)) + np.exp(-(x - d_slit/2)**2 / (2 * sigma**2)))

def compute_visibility(x, I):
    peaks, _ = find_peaks(I, distance=len(x)//30)
    if len(peaks) < 2: return 0.0
    I_max = np.max(I[peaks])
    I_min = np.min(I)
    return (I_max - I_min) / (I_max + I_min) if (I_max + I_min) > 0 else 0

@interact(v_mean=FloatSlider(value=5.8e5, min=2e5, max=1.2e6, description='Velocity'),
          a_width=FloatSlider(value=0.72e-6, min=0.2e-6, max=1.5e-6, description='Slit Width'),
          d_slit=FloatSlider(value=2.45e-6, min=1.0e-6, max=5.0e-6, description='Separation'),
          observer_active=Checkbox(value=False, description='Detector ON'),
          meas_strength=FloatSlider(value=0.0, min=0.0, max=1.0, description='Strength'))
def interactive_lab(v_mean, a_width, d_slit, observer_active, meas_strength):
    x = np.linspace(-0.005, 0.005, 1000)
    I_interf = double_slit_intensity_single_velocity(x, v_mean, L_total, a_width, d_slit)
    I_part = particle_like_pattern(x, v_mean, L_total, a_width, d_slit)
    I = ((1 - meas_strength) * I_interf + meas_strength * I_part) if observer_active else I_interf
    I /= np.max(I)
    
    # حساب الزوايا وعرضها (إضافة ديبسيك)
    (ti_r, ti_d, td_r, td_d) = compute_angles(v_mean, a_width, d_slit, L_total)
    visibility = compute_visibility(x, I)
    
    print(f"✅ Visibility: {visibility:.1%}")
    print(f"📐 First interference max angle: {ti_d:.4f}°")
    print(f"📐 First diffraction min angle: {td_d:.4f}°")

    plt.figure(figsize=(10, 5))
    plt.plot(x*1000, I)
    plt.title('Berramdane Model V9.4 | Quantum Simulation')
    plt.grid(True)
    plt.show()"